# 05 — Model Evaluation (Pre-Hypothesis Testing)

**Primary author:** Victoria

**Builds on:**
- *03_train_g1.ipynb* (Victoria — produced the training and validation triplet files used for triplet accuracy in §2 / §3)
- *04_embedding_verification.ipynb* (Victoria — confirmed all validation embedding sets are shape-correct, non-degenerate, and index-aligned; this notebook relies on those guarantees)
- *notebooks/archive/09_learned_g_misdirection.ipynb* (Nathan — the earlier `g1_tokenspan` analysis whose collapse symptom motivated §4)
- *DECISIONS.md §20* (mean pooling is canonical; `g1` in this notebook refers to the mean-pooled model, not the tokenspan variant)
- *DECISIONS.md §21* (Validation triplet accuracy uses resolvable subset — prior reading)
- *DECISIONS.md §22* (Full-vocabulary wnex embeddings for g_stock and g1)

**Prompt engineering:** Victoria  
**AI assistance:** Claude / Claude Code (Anthropic)  
**Environment:** Local

---

Before we compute and interpret the ATE in the Stage 6 hypothesis test, we need to be sure we understand the basic health of `g1`. This notebook produces five diagnostic readings:

1. **Validation triplet accuracy under wndef (§2).** Does `g1` satisfy the
   training objective on held-out triplets it has never seen? This revision
   uses the *full-vocabulary* `f_common_wndef` embeddings (53,930 words),
   resolving nearly all distractors that the prior val-only version
   (Decision 21) had to drop.
2. **Cross-f triplet accuracy (§3).** Does `g1`'s discriminative structure
   transfer from wndef (the phrase format it was trained on) to wnex (a
   format it has never seen)? This is the central new analysis — a
   matched-comparison generalization test.
3. **Collapse detection (§4).** Mean pairwise cosine among random word
   pairs and the participation ratio of the embedding spectrum, on
   validation-only embeddings. Diagnoses the indiscriminate-compression
   failure mode seen in NB 09.
4. **T=0 and T=1 distributions (§5 wndef, §6 wnex).** Decomposes the two
   components of the ATE so we can see whether a change comes from T=0
   shifting, T=1 shifting, or both. Reported under both wndef and wnex.
5. **RSA (§7).** Single-number measure of how much the pairwise similarity
   structure was reorganized by fine-tuning, on validation-only embeddings.

Everything runs locally on CPU from pre-computed embeddings — no new GPU
work.

**Scope.** Canonical mean-pooling models only (`g_stock` and `g1`). The
`_tokenspan` variants are out of scope per Decision 20.

**Full-vocabulary vs val-only.** Triplet accuracy (§2, §3) uses
*full-vocabulary* wndef and wnex embeddings (53,930 / 8,360 words) because
we need to resolve distractor lookups across the full WordNet vocabulary,
not just words that happen to appear in validation clues. Collapse (§4),
T=0/T=1 (§5, §6), and RSA (§7) use *val-only* embeddings (26,152 / 3,008
words) because these are model-selection diagnostics and must not include
test-split words (Decision 9). T=0/T=1 evaluates only validation clues, so
either choice would yield the same numbers — val-only is used for
consistency with the surrounding sections.

**This is an exploratory pass.** A future revision will tighten the
narrative and trim sections that turn out not to be informative.

---

## §0 — Imports and configuration

Environment auto-detection for Local / Great Lakes / Colab. A version-
reporting cell follows (Decision 18) — this prints rather than asserts so
that a collaborator with a slightly different environment can still run
the notebook, but any mismatch is visible at a glance.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT  = PROJECT_ROOT / "custom_embedding_model"
EMBEDDINGS_DIR  = COMPONENT_ROOT / "data" / "embeddings"
WN_DIR          = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
TRIPLETS_DIR    = COMPONENT_ROOT / "data" / "triplets"
OUTPUT_DIR      = COMPONENT_ROOT / "outputs"
FIG_DIR         = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:     {env_label}")
print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"EMBEDDINGS_DIR:  {EMBEDDINGS_DIR}")
print(f"TRIPLETS_DIR:    {TRIPLETS_DIR}")
print(f"OUTPUT_DIR:      {OUTPUT_DIR}")

# Pin the RNG once. §4 and §7 sample pairs / vocabulary subsets; fixing the
# seed keeps those samples identical across re-runs.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Version reporting (Decision 18). Printed, not asserted.
import matplotlib, seaborn
print()
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"scipy:      {scipy.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn:    {seaborn.__version__}")

---

## §1 — Load embeddings and index files

This notebook reads two distinct sets of embeddings:

- **Full-vocabulary** wndef (53,930 words) and wnex (8,360 words) for both
  models. These are used in §2 and §3 for triplet accuracy. Distractor
  words in `g1_val.csv` are drawn from the full WordNet vocabulary; without
  full-vocab embeddings we drop ~41% of triplets (Decision 21). The
  full-vocab wnex arrays were generated 2026-04-19 specifically to enable
  the cross-f triplet accuracy test in §3 (Decision 22).
- **Validation-only** wndef (26,152 words) and wnex (3,008 words) for both
  models. These are used in §4 (collapse) and §7 (RSA), which are
  model-selection diagnostics that must not include test-split words
  (Decision 9), and in §5 / §6 (T=0/T=1) for consistency with the
  surrounding sections.

We also load the shared `f_clue_val_index.csv` (NB 04 confirmed the two
models' indexes are identical, so we use `g_stock`'s copy as canonical),
the four vocabulary CSVs, `clues_val.csv`, and the validation triplet
file `g1_val.csv`.

All CSVs load with `keep_default_na=False, na_values=[""]` because `"nan"`
(grandmother) is a valid crossword word and must not be coerced to NaN.

Embeddings are stored in a single dict keyed by `(model, phrase)` so later
sections can index them uniformly. We deliberately keep **separate** word
lookup dicts for full-vocab and val-only vocabularies (e.g.
`wndef_word_to_row` vs `wndef_val_word_to_row`) to make accidental
cross-use — looking up a word in the wrong-sized embedding array —
impossible by construction.

In [ ]:
# ============================================================
# Model / phrase-type registry
# ============================================================
MODEL_NAMES = ["g_stock", "g1"]

# Phrase keys in the embeddings dict. Full-vocab keys carry no _val suffix,
# val-only keys carry _val — matching the on-disk filename convention.
PHRASE_KEYS = [
    "f_clue_val",
    "f_common_wndef",
    "f_common_wnex",
    "f_common_wndef_val",
    "f_common_wnex_val",
]

EXPECTED_SHAPES = {
    "f_clue_val":         (47933, 1024),
    "f_common_wndef":     (53930, 1024),
    "f_common_wnex":      ( 8360, 1024),
    "f_common_wndef_val": (26152, 1024),
    "f_common_wnex_val":  ( 3008, 1024),
}

In [ ]:
# ============================================================
# Load all .npy arrays + index / vocabulary / clue / triplet CSVs
# ============================================================
t0 = time.time()

# embeddings[(model, phrase)] -> np.ndarray of shape (N, 1024)
embeddings = {}
for model in MODEL_NAMES:
    for phrase in PHRASE_KEYS:
        emb = np.load(EMBEDDINGS_DIR / model / f"{phrase}.npy")
        # Shape validation per CLAUDE.md — catches any accidental swap of one
        # phrase file for another (different row counts).
        assert emb.shape == EXPECTED_SHAPES[phrase], (
            f"{model}/{phrase}: shape {emb.shape} != expected {EXPECTED_SHAPES[phrase]}"
        )
        embeddings[(model, phrase)] = emb

# Shared f_clue_val index (identical across models per NB 04's consistency check).
f_clue_index = pd.read_csv(
    EMBEDDINGS_DIR / "g_stock" / "f_clue_val_index.csv",
    keep_default_na=False, na_values=[""],
)

# Vocabulary files — row column is the canonical index into the corresponding
# .npy array. Full-vocab files index full-vocab embeddings; val-only files
# index val-only embeddings.
vocab_wndef = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wnex = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wndef_val = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef_val.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wnex_val = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex_val.csv",
    keep_default_na=False, na_values=[""],
)

# Validation clue rows, for the (clue, definition, answer) eval pairs in §5/§6.
clues_val = pd.read_csv(
    WN_DIR / "clues_val.csv",
    keep_default_na=False, na_values=[""],
)

# Validation triplets, for §2 and §3.
val_triplets = pd.read_csv(
    TRIPLETS_DIR / "g1_val.csv",
    keep_default_na=False, na_values=[""],
)

load_seconds = time.time() - t0
print(f"Loaded {len(MODEL_NAMES) * len(PHRASE_KEYS)} .npy + 6 CSVs in {load_seconds:.1f}s")
print(f"  vocabulary_wndef       (full):     {len(vocab_wndef):,} rows")
print(f"  vocabulary_wnex        (full):     {len(vocab_wnex):,} rows")
print(f"  vocabulary_wndef_val:              {len(vocab_wndef_val):,} rows")
print(f"  vocabulary_wnex_val:               {len(vocab_wnex_val):,} rows")
print(f"  f_clue_val_index:                  {len(f_clue_index):,} rows")
print(f"  clues_val:                         {len(clues_val):,} rows")
print(f"  val_triplets (g1_val.csv):         {len(val_triplets):,} rows")

# Cross-check .npy row counts against the corresponding vocabulary / index
# file lengths, separate from the EXPECTED_SHAPES constant check above.
for model in MODEL_NAMES:
    assert embeddings[(model, "f_clue_val")].shape[0]         == len(f_clue_index)
    assert embeddings[(model, "f_common_wndef")].shape[0]     == len(vocab_wndef)
    assert embeddings[(model, "f_common_wnex")].shape[0]      == len(vocab_wnex)
    assert embeddings[(model, "f_common_wndef_val")].shape[0] == len(vocab_wndef_val)
    assert embeddings[(model, "f_common_wnex_val")].shape[0]  == len(vocab_wnex_val)
print("All row-count cross-checks pass.")

In [ ]:
# ============================================================
# Lookup dicts: word / clue-key -> row index
# ============================================================
# Building these once here avoids re-scanning CSVs in every later section.
# Full-vocab and val-only dicts are kept separate by name so a section that
# uses one cannot accidentally index the wrong embedding array.
wndef_word_to_row     = dict(zip(vocab_wndef["word"],     vocab_wndef["row"]))
wnex_word_to_row      = dict(zip(vocab_wnex["word"],      vocab_wnex["row"]))
wndef_val_word_to_row = dict(zip(vocab_wndef_val["word"], vocab_wndef_val["row"]))
wnex_val_word_to_row  = dict(zip(vocab_wnex_val["word"],  vocab_wnex_val["row"]))

# f_clue is indexed by the composite (clue_id, definition) key — use it as a tuple.
clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        f_clue_index["clue_id"], f_clue_index["definition"], f_clue_index["row"]
    )
}

print(f"wndef_word_to_row     (full):     {len(wndef_word_to_row):,} entries")
print(f"wnex_word_to_row      (full):     {len(wnex_word_to_row):,} entries")
print(f"wndef_val_word_to_row:            {len(wndef_val_word_to_row):,} entries")
print(f"wnex_val_word_to_row:             {len(wnex_val_word_to_row):,} entries")
print(f"clue_key_to_row:                  {len(clue_key_to_row):,} entries")

In [ ]:
# ============================================================
# Rowwise cosine helper (matches the DATA.md definition)
# ============================================================
def rowwise_cosine(A, B):
    """Return per-row cosine similarity between two equal-shape (N, D) arrays.

    Identical to the helper used in NB 04 and documented in DATA.md.
    """
    # +1e-10 guard against accidental zero rows. NB 04 already checked for
    # zero rows in all our arrays, so this should never fire.
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

---

## §2 — Validation triplet accuracy (wndef, full-vocabulary)

This is the most direct overfitting diagnostic we have: the triplet
constraint the model was trained on, evaluated on triplets the model has
never seen. `data/triplets/g1_val.csv` was constructed in NB 03 using the
identical pipeline to `g1_train.csv` — same distractor source, same phrase
lookups — so differences in triplet accuracy between train and validate
reflect generalization, not construction artifacts.

**What's new in this revision.** The previous run (Decision 21) used
validation-only wndef embeddings (26,152 words), which dropped ~41% of
triplets because their distractors weren't in the val vocabulary.
Full-vocabulary wndef embeddings (53,930 words) resolve nearly all
distractors — only those that fall completely outside our wn_synset
vocabulary (per `g1_train_meta.json`, ~222 distinct words across all
splits) should still fail. We expect ~46,200+ of the 46,506 triplets to
resolve cleanly here.

For each validation triplet we resolve three row indices (anchor,
positive, negative), pull the corresponding pre-computed embeddings, and
check whether `cos(anchor, positive) > cos(anchor, negative)`. We report
this both for `g1` and for `g_stock`; a large gap `g1 − g_stock` is
evidence the fine-tuning learned a generalizable signal.

In [ ]:
# ============================================================
# §2a — Resolve triplet embedding rows (full-vocab wndef)
# ============================================================
print(f"Loaded {len(val_triplets):,} validation triplets from g1_val.csv")
print(f"Columns: {list(val_triplets.columns)}")

# Resolve row indices for each of the three triplet roles. A miss returns -1
# so we can vectorize the drop step. Positives and negatives are looked up in
# the FULL-vocab wndef dict (53,930 words), not the val-only one.
anchor_rows = np.array([
    clue_key_to_row.get((cid, defn), -1)
    for cid, defn in zip(val_triplets["clue_id"], val_triplets["definition"])
])
positive_rows_wndef = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["answer_wn"]
])
negative_rows_wndef = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["distractor_wn"]
])

n_total = len(val_triplets)
n_miss_anchor   = int((anchor_rows         < 0).sum())
n_miss_positive = int((positive_rows_wndef < 0).sum())
n_miss_negative = int((negative_rows_wndef < 0).sum())

print(f"\nResolution counts (full-vocab wndef):")
print(f"  anchors resolved:   {n_total - n_miss_anchor:,} / {n_total:,}")
print(f"  positives resolved: {n_total - n_miss_positive:,} / {n_total:,}")
print(f"  negatives resolved: {n_total - n_miss_negative:,} / {n_total:,}")

# Anchors and positives must always resolve: anchors come from validation-split
# clues (fully covered by f_clue_val), and positives are answer_wn values which
# are by construction in the wndef vocabulary.
assert n_miss_anchor == 0,   f"{n_miss_anchor:,} anchors failed to resolve"
assert n_miss_positive == 0, f"{n_miss_positive:,} positives failed to resolve"

mask_wndef = (negative_rows_wndef >= 0)
n_resolved_wndef = int(mask_wndef.sum())
print(f"  all three resolved: {n_resolved_wndef:,} / {n_total:,} "
      f"({n_resolved_wndef / n_total:.1%})")
print(f"  expected (per g1_train_meta.json): ~46,200+ — only ~222 distinct "
      f"distractor words fall outside the wndef vocabulary")

# Slice the row arrays to the resolved subset for use in §2b/§2c. The full
# arrays remain available for §3's matched-subset selection (which uses the
# wnex vocabulary as the binding constraint).
anchor_rows_wndef = anchor_rows[mask_wndef]
pos_rows_wndef    = positive_rows_wndef[mask_wndef]
neg_rows_wndef    = negative_rows_wndef[mask_wndef]

In [ ]:
# ============================================================
# §2b — Triplet accuracy table (g_stock vs g1, wndef)
# ============================================================
def triplet_stats(model, anchor_rows, pos_rows, neg_rows, pos_neg_phrase):
    """Compute triplet accuracy stats for one (model, phrase_type) combination.

    Parameters
    ----------
    model : str
        Model name key in `embeddings` (e.g. "g_stock", "g1").
    anchor_rows : np.ndarray
        Int array of f_clue_val row indices.
    pos_rows, neg_rows : np.ndarray
        Int arrays of vocabulary row indices for positives / negatives.
        Must be valid indices into the array stored at
        `embeddings[(model, pos_neg_phrase)]`.
    pos_neg_phrase : str
        Phrase key for the positive/negative embeddings
        (e.g. "f_common_wndef" or "f_common_wnex").

    Returns
    -------
    dict with summary statistics plus the raw `_margin` vector for plotting.
    """
    clue_emb = embeddings[(model, "f_clue_val")]
    pn_emb   = embeddings[(model, pos_neg_phrase)]
    A = clue_emb[anchor_rows]
    P = pn_emb[pos_rows]
    N = pn_emb[neg_rows]
    cos_pos = rowwise_cosine(A, P)
    cos_neg = rowwise_cosine(A, N)
    margin  = cos_pos - cos_neg
    return {
        "model":         model,
        "phrase":        pos_neg_phrase,
        "n_triplets":    len(margin),
        "accuracy":      float((margin > 0).mean()),
        "mean_margin":   float(margin.mean()),
        "median_margin": float(np.median(margin)),
        "pct_margin_over_0.1": float((margin > 0.1).mean()),
        "pct_margin_over_0.5": float((margin > 0.5).mean()),
        "_margin":  margin,
        "_cos_pos": cos_pos,
        "_cos_neg": cos_neg,
    }

stats_stock_wndef = triplet_stats(
    "g_stock", anchor_rows_wndef, pos_rows_wndef, neg_rows_wndef, "f_common_wndef",
)
stats_g1_wndef = triplet_stats(
    "g1", anchor_rows_wndef, pos_rows_wndef, neg_rows_wndef, "f_common_wndef",
)

triplet_table = pd.DataFrame([
    {"Metric": "Triplet accuracy (% correct)",
     "g_stock": stats_stock_wndef["accuracy"] * 100,
     "g1":      stats_g1_wndef["accuracy"]    * 100},
    {"Metric": "Mean margin (cos_pos - cos_neg)",
     "g_stock": stats_stock_wndef["mean_margin"],
     "g1":      stats_g1_wndef["mean_margin"]},
    {"Metric": "Median margin",
     "g_stock": stats_stock_wndef["median_margin"],
     "g1":      stats_g1_wndef["median_margin"]},
    {"Metric": "% triplets with margin > 0.1",
     "g_stock": stats_stock_wndef["pct_margin_over_0.1"] * 100,
     "g1":      stats_g1_wndef["pct_margin_over_0.1"]    * 100},
    {"Metric": "% triplets with margin > 0.5",
     "g_stock": stats_stock_wndef["pct_margin_over_0.5"] * 100,
     "g1":      stats_g1_wndef["pct_margin_over_0.5"]    * 100},
    {"Metric": "N validation triplets evaluated",
     "g_stock": stats_stock_wndef["n_triplets"],
     "g1":      stats_g1_wndef["n_triplets"]},
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(triplet_table.to_string(index=False))

In [ ]:
# ============================================================
# §2c — Margin distribution histogram (full-vocab wndef)
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))

# Use a common bin edge set so the two histograms are directly comparable.
both = np.concatenate([stats_stock_wndef["_margin"], stats_g1_wndef["_margin"]])
bins = np.linspace(both.min(), both.max(), 80)

ax.hist(stats_stock_wndef["_margin"], bins=bins, alpha=0.55,
        label="g_stock", color="tab:blue")
ax.hist(stats_g1_wndef["_margin"],    bins=bins, alpha=0.55,
        label="g1",      color="tab:orange")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--",
           label="margin = 0 (correct / incorrect boundary)")
ax.axvline(stats_stock_wndef["_margin"].mean(), color="tab:blue",
           linewidth=1.2, linestyle=":")
ax.axvline(stats_g1_wndef["_margin"].mean(),    color="tab:orange",
           linewidth=1.2, linestyle=":")
ax.set_xlabel("cos(anchor, positive) - cos(anchor, negative)")
ax.set_ylabel("Count of validation triplets")
ax.set_title(
    f"Validation triplet margin distribution — wndef (full vocab, "
    f"{stats_g1_wndef['n_triplets']:,} triplets)"
)
ax.legend()
fig.tight_layout()

fig_path = FIG_DIR / "05_val_triplet_accuracy.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §3 — Cross-f triplet accuracy (matched comparison)

This is the central new analysis. `g1` was trained on triplets where the
positive and negative embeddings come from `f_common_wndef`
(`"<t>word</t>: <WordNet definition>"`). Does the discriminative structure
it learned generalize to a different phrase format — `f_common_wnex`
(`"<t>word</t> appearing once inside a usage example sentence"`) — that the
model has never been exposed to during training?

**Methodology.** For a fair comparison, we evaluate the *exact same* set
of triplets under both wndef and wnex embeddings. The binding constraint
is the wnex vocabulary (8,360 words) since wnex is a strict subset of wndef
(per FINDINGS.md / Stage 2 cross-f comparison). A triplet enters the
matched set if both its `answer_wn` (positive) and its `distractor_wn`
(negative) appear in `vocabulary_wnex.csv`. Anchors always resolve
via `f_clue_val` regardless of phrase format.

Two key comparisons fall out of the resulting table:

- **g1 wndef vs g1 wnex** — does the triplet accuracy g1 achieves on
  wndef-format triplets transfer to wnex-format triplets on the *same*
  underlying word triples?
- **g1 wnex vs g_stock wnex** — does g1 improve over the untrained baseline
  on wnex format, despite never seeing wnex phrases during training?

In [ ]:
# ============================================================
# §3a — Identify the matched subset (positives AND negatives in wnex)
# ============================================================
# Resolve positives and negatives against the FULL wnex vocabulary.
positive_rows_wnex = np.array([
    wnex_word_to_row.get(w, -1) for w in val_triplets["answer_wn"]
])
negative_rows_wnex = np.array([
    wnex_word_to_row.get(w, -1) for w in val_triplets["distractor_wn"]
])

# Per-role wnex hit/miss to characterize the dropout.
pos_in_wnex = positive_rows_wnex >= 0
neg_in_wnex = negative_rows_wnex >= 0

n_both_in   = int(( pos_in_wnex &  neg_in_wnex).sum())
n_pos_only  = int(( pos_in_wnex & ~neg_in_wnex).sum())
n_neg_only  = int((~pos_in_wnex &  neg_in_wnex).sum())
n_neither   = int((~pos_in_wnex & ~neg_in_wnex).sum())

print(f"Wnex resolution breakdown across {n_total:,} val triplets:")
print(f"  both answer_wn and distractor_wn in wnex: {n_both_in:,} "
      f"({n_both_in / n_total:.1%}) — the matched subset")
print(f"  answer in wnex, distractor not:           {n_pos_only:,} "
      f"({n_pos_only / n_total:.1%})")
print(f"  distractor in wnex, answer not:           {n_neg_only:,} "
      f"({n_neg_only / n_total:.1%})")
print(f"  neither in wnex:                          {n_neither:,} "
      f"({n_neither / n_total:.1%})")

# The matched subset itself — anchors automatically resolve, positives and
# negatives must both be in wnex (which is a strict subset of wndef per
# FINDINGS.md, so they're automatically in wndef too).
matched_mask = pos_in_wnex & neg_in_wnex
anchor_rows_matched = anchor_rows[matched_mask]
pos_rows_matched_wndef = positive_rows_wndef[matched_mask]
neg_rows_matched_wndef = negative_rows_wndef[matched_mask]
pos_rows_matched_wnex  = positive_rows_wnex[matched_mask]
neg_rows_matched_wnex  = negative_rows_wnex[matched_mask]
n_matched = int(matched_mask.sum())
print(f"\nMatched subset size: {n_matched:,} triplets "
      f"({n_matched / n_total:.1%} of {n_total:,})")

In [ ]:
# ============================================================
# §3b — Matched triplet accuracy table (4 combinations)
# ============================================================
# Run triplet_stats once per (model, phrase) on the SAME matched subset of
# triplets. The wndef and wnex calls share anchor rows (and the underlying
# val_triplets row indices) but use different positive/negative row arrays
# and different positive/negative embedding arrays.
stats_stock_wndef_m = triplet_stats(
    "g_stock", anchor_rows_matched,
    pos_rows_matched_wndef, neg_rows_matched_wndef, "f_common_wndef",
)
stats_g1_wndef_m = triplet_stats(
    "g1", anchor_rows_matched,
    pos_rows_matched_wndef, neg_rows_matched_wndef, "f_common_wndef",
)
stats_stock_wnex_m = triplet_stats(
    "g_stock", anchor_rows_matched,
    pos_rows_matched_wnex, neg_rows_matched_wnex, "f_common_wnex",
)
stats_g1_wnex_m = triplet_stats(
    "g1", anchor_rows_matched,
    pos_rows_matched_wnex, neg_rows_matched_wnex, "f_common_wnex",
)

crossf_table = pd.DataFrame([
    {"Metric": "Triplet accuracy (% correct)",
     "g_stock (wndef)": stats_stock_wndef_m["accuracy"]    * 100,
     "g1 (wndef)":      stats_g1_wndef_m["accuracy"]       * 100,
     "g_stock (wnex)":  stats_stock_wnex_m["accuracy"]     * 100,
     "g1 (wnex)":       stats_g1_wnex_m["accuracy"]        * 100},
    {"Metric": "Mean margin (cos_pos - cos_neg)",
     "g_stock (wndef)": stats_stock_wndef_m["mean_margin"],
     "g1 (wndef)":      stats_g1_wndef_m["mean_margin"],
     "g_stock (wnex)":  stats_stock_wnex_m["mean_margin"],
     "g1 (wnex)":       stats_g1_wnex_m["mean_margin"]},
    {"Metric": "Median margin",
     "g_stock (wndef)": stats_stock_wndef_m["median_margin"],
     "g1 (wndef)":      stats_g1_wndef_m["median_margin"],
     "g_stock (wnex)":  stats_stock_wnex_m["median_margin"],
     "g1 (wnex)":       stats_g1_wnex_m["median_margin"]},
    {"Metric": "% triplets with margin > 0.1",
     "g_stock (wndef)": stats_stock_wndef_m["pct_margin_over_0.1"] * 100,
     "g1 (wndef)":      stats_g1_wndef_m["pct_margin_over_0.1"]    * 100,
     "g_stock (wnex)":  stats_stock_wnex_m["pct_margin_over_0.1"]  * 100,
     "g1 (wnex)":       stats_g1_wnex_m["pct_margin_over_0.1"]     * 100},
    {"Metric": "N triplets",
     "g_stock (wndef)": stats_stock_wndef_m["n_triplets"],
     "g1 (wndef)":      stats_g1_wndef_m["n_triplets"],
     "g_stock (wnex)":  stats_stock_wnex_m["n_triplets"],
     "g1 (wnex)":       stats_g1_wnex_m["n_triplets"]},
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 160):
    print(crossf_table.to_string(index=False))

In [ ]:
# ============================================================
# §3c — Cross-f margin distribution figure (2 panels)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

panels = [
    (axes[0], "wndef (matched subset)", stats_stock_wndef_m, stats_g1_wndef_m),
    (axes[1], "wnex  (matched subset)", stats_stock_wnex_m,  stats_g1_wnex_m),
]
for ax, label, s_stock, s_g1 in panels:
    # Common bins per panel so the g_stock and g1 distributions overlay correctly.
    both = np.concatenate([s_stock["_margin"], s_g1["_margin"]])
    bins = np.linspace(both.min(), both.max(), 70)
    ax.hist(s_stock["_margin"], bins=bins, alpha=0.55,
            label=f"g_stock (mean={s_stock['_margin'].mean():.3f})",
            color="tab:blue")
    ax.hist(s_g1["_margin"],    bins=bins, alpha=0.55,
            label=f"g1 (mean={s_g1['_margin'].mean():.3f})",
            color="tab:orange")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(s_stock["_margin"].mean(), color="tab:blue",
               linewidth=1.2, linestyle=":")
    ax.axvline(s_g1["_margin"].mean(),    color="tab:orange",
               linewidth=1.2, linestyle=":")
    ax.set_title(f"Margin distribution — {label}")
    ax.set_xlabel("cos(anchor, positive) - cos(anchor, negative)")
    ax.set_ylabel("count")
    ax.legend()
fig.suptitle(
    f"Cross-f triplet accuracy on matched subset ({n_matched:,} triplets)"
)
fig.tight_layout()
fig_path = FIG_DIR / "05_crossf_triplet_accuracy.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §4 — Collapse detection (val-only)

A healthy embedding space discriminates: semantically distinct words sit
far apart, semantically related ones close together. A collapsed space
pulls everything together indiscriminately — the geometry compresses and
cosine-based downstream tasks lose their signal.

We test for collapse two ways:

1. **Mean pairwise cosine among random word pairs.** If `g1` compresses
   the space, arbitrary pairs should look more similar under `g1` than
   under `g_stock`.
2. **Effective dimensionality (participation ratio).** A collapsed space
   concentrates variance into a few principal directions; a healthy space
   spreads it across many. The participation ratio `(Σs²)² / Σs⁴` maps
   that intuition onto a single number bounded between 1 (maximally
   collapsed) and D (perfectly isotropic).

**Why val-only here?** Collapse is a model-selection diagnostic. Loading
test-split words into the diagnostic would expose us to test data during
iteration, violating the spirit of Decision 9. The val-only embeddings
(26,152 wndef, 3,008 wnex) are sufficient for measuring spectral
properties of the learned representation.

In [ ]:
# ============================================================
# §4a — Mean pairwise cosine among random word pairs (val-only)
# ============================================================
# Use the same sampled pair indices for both models within a phrase type so
# the two numbers are directly comparable.
N_PAIRS = 50_000

def sample_pairs(n_rows, n_pairs, seed):
    """Return two int arrays (i, j) of length n_pairs, with i != j on every row.

    Sampling with replacement from the product is fine for 50k pairs out of
    at least C(3008, 2) = ~4.5M candidates — the collision rate is negligible,
    and any stray (i, i) pair gets resampled below.
    """
    rng = np.random.default_rng(seed)
    i = rng.integers(0, n_rows, size=n_pairs)
    j = rng.integers(0, n_rows, size=n_pairs)
    # Resample any self-pair until clean. In practice this resolves in one pass.
    same = i == j
    while same.any():
        j[same] = rng.integers(0, n_rows, size=int(same.sum()))
        same = i == j
    return i, j

pairwise_rows = []
pair_sims = {}  # keyed by (model, phrase) for §4c's histogram
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    # Same (i, j) pairs across the two models for this phrase type.
    n_rows_phrase = embeddings[(MODEL_NAMES[0], phrase)].shape[0]
    i_idx, j_idx = sample_pairs(n_rows_phrase, N_PAIRS, seed=RANDOM_STATE)

    for model in MODEL_NAMES:
        emb = embeddings[(model, phrase)]
        A = emb[i_idx]
        B = emb[j_idx]
        sims = rowwise_cosine(A, B)
        pair_sims[(model, phrase)] = sims
        pairwise_rows.append({
            "Model":       model,
            "Phrase type": phrase,
            "Mean":        float(sims.mean()),
            "Median":      float(np.median(sims)),
            "Std":         float(sims.std()),
            "P5":          float(np.percentile(sims, 5)),
            "P95":         float(np.percentile(sims, 95)),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(pairwise_df.to_string(index=False))

In [ ]:
# ============================================================
# §4b — Embedding variance and effective dimensionality (val-only)
# ============================================================
def variance_stats(emb):
    """Total variance, participation ratio, and cumulative variance fractions.

    We center the embedding matrix and take its singular values. Squared
    singular values are proportional to variance per principal component.
    Participation ratio = (sum(s^2))^2 / sum(s^4) — 1 when all variance sits
    in a single component, D when it is perfectly spread across D dims.
    """
    centered = emb - emb.mean(axis=0, keepdims=True)
    # full_matrices=False keeps SVD in the economy form — still exact, much cheaper.
    s = np.linalg.svd(centered, full_matrices=False, compute_uv=False)
    s2 = s ** 2
    total_var = float(s2.sum())
    eff_dim = float(total_var ** 2 / np.sum(s ** 4))
    cum = np.cumsum(s2) / total_var
    return {
        "total_var":   total_var,
        "eff_dim":     eff_dim,
        "top10_frac":  float(cum[9]),
        "top50_frac":  float(cum[49]),
        "top100_frac": float(cum[99]),
        "_cum": cum,
    }

variance_rows = []
variance_detail = {}  # (model, phrase) -> dict incl. _cum for §4c
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    for model in MODEL_NAMES:
        stats = variance_stats(embeddings[(model, phrase)])
        variance_detail[(model, phrase)] = stats
        variance_rows.append({
            "Model":         model,
            "Phrase type":   phrase,
            "Total var":     stats["total_var"],
            "Eff. dim":      stats["eff_dim"],
            "Top-10 var %":  stats["top10_frac"]  * 100,
            "Top-50 var %":  stats["top50_frac"]  * 100,
            "Top-100 var %": stats["top100_frac"] * 100,
        })

variance_df = pd.DataFrame(variance_rows)
with pd.option_context("display.float_format", "{:.2f}".format,
                       "display.width", 140):
    print(variance_df.to_string(index=False))

In [ ]:
# ============================================================
# §4c — Collapse visualization 1: pairwise cosine histograms
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, phrase in zip(axes, ["f_common_wndef_val", "f_common_wnex_val"]):
    # Common bins so g_stock and g1 histograms are directly comparable.
    both = np.concatenate([pair_sims[("g_stock", phrase)],
                           pair_sims[("g1",      phrase)]])
    bins = np.linspace(both.min(), both.max(), 70)
    for model, color in [("g_stock", "tab:blue"), ("g1", "tab:orange")]:
        sims = pair_sims[(model, phrase)]
        ax.hist(sims, bins=bins, alpha=0.55, color=color,
                label=f"{model} (mean={sims.mean():.3f})")
    ax.set_title(f"Random pairwise cosine — {phrase}")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("count")
    ax.legend()
fig.suptitle(f"Pairwise cosine among {N_PAIRS:,} random word pairs (val-only)")
fig.tight_layout()
fig_path = FIG_DIR / "05_collapse_pairwise_cosine.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

In [ ]:
# ============================================================
# §4c (cont.) — Collapse visualization 2: cumulative variance curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, phrase in zip(axes, ["f_common_wndef_val", "f_common_wnex_val"]):
    for model, color in [("g_stock", "tab:blue"), ("g1", "tab:orange")]:
        cum = variance_detail[(model, phrase)]["_cum"]
        eff = variance_detail[(model, phrase)]["eff_dim"]
        ax.plot(np.arange(1, len(cum) + 1), cum, color=color,
                label=f"{model} (eff. dim={eff:.1f})")
    ax.set_xscale("log")
    ax.set_title(f"Cumulative variance — {phrase}")
    ax.set_xlabel("number of components (log scale)")
    ax.set_ylabel("fraction of total variance")
    ax.set_ylim(0, 1.02)
    ax.axhline(1.0, color="grey", linewidth=0.5)
    ax.legend(loc="lower right")
fig.suptitle("Effective dimensionality: how many components carry the variance? (val-only)")
fig.tight_layout()
fig_path = FIG_DIR / "05_collapse_singular_values.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §5 — T=0 and T=1 similarity distributions (wndef)

The ATE is a difference:
`ATE = mean( cos(g(f_clue(def)), g(f(ans))) − cos(g(f(def)), g(f(ans))) )`.

Looking only at the difference hides whether a change in ATE comes from
T=0 shifting, T=1 shifting, or both. In the NB 09 analysis we saw T=0
(the decontextualized similarity) rise while T=1 stayed roughly put — the
symptom of indiscriminate compression. Here we compute T=0 and T=1
separately for every evaluation pair under wndef and compare distributions.

**Evaluation pairs.** For each row of `clues_val.csv` we need three
lookups: the clue's `f_clue_val` row, the `f_common_wndef_val` row for
`definition_wn`, and the `f_common_wndef_val` row for `answer_wn`. Since
`clues_val.csv` rows come from the validation split, every word in
`definition_wn` and `answer_wn` is by construction in
`vocabulary_wndef_val.csv`, so we expect 100% resolution.

Val-only embeddings are used here for consistency with §4 and §7 —
evaluation pairs come from validation clues and resolve fully under either
the val-only or the full vocabulary.

In [ ]:
# ============================================================
# §5a — Assemble (clue, definition, answer) eval pairs (wndef val)
# ============================================================
t0 = time.time()

clue_rows_out = []
def_rows_out  = []
ans_rows_out  = []

# Separate counts for each dropout cause, to distinguish "definition missing"
# from "answer missing" from "clue missing from index".
n_eval_total       = len(clues_val)
n_missing_clue     = 0
n_missing_def_wndef = 0
n_missing_ans_wndef = 0

for cid, original_def, def_wn, ans_wn in zip(
    clues_val["clue_id"], clues_val["definition"],
    clues_val["definition_wn"], clues_val["answer_wn"]
):
    # f_clue is indexed by the ORIGINAL definition string (not definition_wn),
    # matching how f_clue.csv was built in NB 02.
    clue_row = clue_key_to_row.get((cid, original_def), -1)
    def_row  = wndef_val_word_to_row.get(def_wn, -1)
    ans_row  = wndef_val_word_to_row.get(ans_wn, -1)
    if clue_row < 0:
        n_missing_clue += 1
        continue
    if def_row < 0:
        n_missing_def_wndef += 1
        continue
    if ans_row < 0:
        n_missing_ans_wndef += 1
        continue
    clue_rows_out.append(clue_row)
    def_rows_out.append(def_row)
    ans_rows_out.append(ans_row)

clue_rows_arr_wndef = np.array(clue_rows_out, dtype=np.int64)
def_rows_arr_wndef  = np.array(def_rows_out,  dtype=np.int64)
ans_rows_arr_wndef  = np.array(ans_rows_out,  dtype=np.int64)
n_kept_wndef = len(clue_rows_arr_wndef)
assemble_seconds = time.time() - t0

print(f"clues_val rows:                                       {n_eval_total:,}")
print(f"  dropped (clue_id, definition) not in f_clue index:  {n_missing_clue:,}")
print(f"  dropped (definition_wn not in vocab_wndef_val):     {n_missing_def_wndef:,}")
print(f"  dropped (answer_wn not in vocab_wndef_val):         {n_missing_ans_wndef:,}")
print(f"  kept (all three lookups resolved):                  {n_kept_wndef:,} "
      f"({n_kept_wndef / n_eval_total:.1%})")
print(f"Assembly: {assemble_seconds:.1f}s")

In [ ]:
# ============================================================
# §5b — T=0, T=1 distributions and ATE preview (wndef val)
# ============================================================
def t0_t1(model, clue_rows, def_rows, ans_rows, vocab_phrase):
    """Compute T=0 and T=1 per evaluation pair under one model.

    `vocab_phrase` selects the phrase format for the decontextualized
    embeddings (e.g. "f_common_wndef_val" or "f_common_wnex_val"). The
    f_clue side is always f_clue_val.
    """
    clue_emb  = embeddings[(model, "f_clue_val")]
    vocab_emb = embeddings[(model, vocab_phrase)]

    def_embeds  = vocab_emb[def_rows]
    ans_embeds  = vocab_emb[ans_rows]
    clue_embeds = clue_emb[clue_rows]

    # T=0: decontextualized definition vs decontextualized answer.
    T0 = rowwise_cosine(def_embeds, ans_embeds)
    # T=1: clue-contextualized definition vs decontextualized answer.
    T1 = rowwise_cosine(clue_embeds, ans_embeds)
    return T0, T1

stock_T0_wndef, stock_T1_wndef = t0_t1(
    "g_stock", clue_rows_arr_wndef, def_rows_arr_wndef, ans_rows_arr_wndef,
    "f_common_wndef_val",
)
g1_T0_wndef, g1_T1_wndef = t0_t1(
    "g1", clue_rows_arr_wndef, def_rows_arr_wndef, ans_rows_arr_wndef,
    "f_common_wndef_val",
)

def dist_stats(name, vec):
    return {
        "Distribution": name,
        "Mean":   float(vec.mean()),
        "Median": float(np.median(vec)),
        "Std":    float(vec.std()),
        "P5":     float(np.percentile(vec, 5)),
        "P95":    float(np.percentile(vec, 95)),
    }

t0_t1_wndef_table = pd.DataFrame([
    dist_stats("g_stock T=0", stock_T0_wndef),
    dist_stats("g_stock T=1", stock_T1_wndef),
    dist_stats("g1 T=0",      g1_T0_wndef),
    dist_stats("g1 T=1",      g1_T1_wndef),
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(t0_t1_wndef_table.to_string(index=False))

ate_stock_wndef = float((stock_T1_wndef - stock_T0_wndef).mean())
ate_g1_wndef    = float((g1_T1_wndef    - g1_T0_wndef   ).mean())
print(f"\nATE preview (wndef, mean of T=1 - T=0 — formal test deferred to Stage 6):")
print(f"  g_stock ATE: {ate_stock_wndef:+.4f}")
print(f"  g1 ATE:      {ate_g1_wndef:+.4f}")

In [ ]:
# ============================================================
# §5c — T=0 / T=1 distribution figure (wndef)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

# Shared bin edges across all four distributions so the histograms are
# directly comparable both within and across panels.
all_vals = np.concatenate([stock_T0_wndef, stock_T1_wndef,
                           g1_T0_wndef,    g1_T1_wndef])
bins = np.linspace(all_vals.min(), all_vals.max(), 70)

panels = [
    (axes[0], "g_stock", stock_T0_wndef, stock_T1_wndef),
    (axes[1], "g1",      g1_T0_wndef,    g1_T1_wndef),
]
for ax, model, T0, T1 in panels:
    ax.hist(T0, bins=bins, alpha=0.55, color="tab:green",
            label=f"T=0 (def vs ans), mean={T0.mean():.3f}")
    ax.hist(T1, bins=bins, alpha=0.55, color="tab:purple",
            label=f"T=1 (clue-def vs ans), mean={T1.mean():.3f}")
    ax.axvline(T0.mean(), color="tab:green",  linewidth=1.2, linestyle=":")
    ax.axvline(T1.mean(), color="tab:purple", linewidth=1.2, linestyle=":")
    ax.set_title(f"{model} — T=0 vs T=1 (wndef, {n_kept_wndef:,} eval pairs)")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("count")
    ax.legend()
fig.tight_layout()
fig_path = FIG_DIR / "05_t0_t1_wndef_distributions.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §6 — T=0 and T=1 similarity distributions (wnex)

Same analysis as §5 but using `f_common_wnex_val` for the decontextualized
embeddings:

- T=0_wnex = `cos(g(f_common_wnex(def)), g(f_common_wnex(ans)))`
- T=1_wnex = `cos(g(f_clue(def)),         g(f_common_wnex(ans)))`

**Eligibility.** A `clues_val.csv` row enters this section's evaluation
set only if BOTH its `definition_wn` and its `answer_wn` are present in
`vocabulary_wnex_val.csv`. wnex covers only ~12% of val vocabulary words
(3,008 of 26,152), so this set will be substantially smaller than §5's
47,933 pairs.

**Note on comparability.** The wnex evaluation set is a subset of §5's
wndef evaluation set, but not a strict alignment of evaluation pairs.
§6d below builds the matched intersection so the wndef-vs-wnex ATE
comparison can be read from the same underlying clue pairs.

In [ ]:
# ============================================================
# §6a — Assemble wnex evaluation pairs from clues_val
# ============================================================
t0 = time.time()

# Per-row eligibility flags so we can report all four diagnostic counts
# (def-in-wnex, ans-in-wnex, both-in-wnex) in addition to the kept count.
n_def_in_wnex = 0
n_ans_in_wnex = 0

clue_rows_out = []
def_rows_out  = []
ans_rows_out  = []
kept_eval_idx = []  # row indices into clues_val of kept pairs (for §6d intersection)

for i, (cid, original_def, def_wn, ans_wn) in enumerate(zip(
    clues_val["clue_id"], clues_val["definition"],
    clues_val["definition_wn"], clues_val["answer_wn"]
)):
    def_in_wnex = def_wn in wnex_val_word_to_row
    ans_in_wnex = ans_wn in wnex_val_word_to_row
    if def_in_wnex:
        n_def_in_wnex += 1
    if ans_in_wnex:
        n_ans_in_wnex += 1
    if not (def_in_wnex and ans_in_wnex):
        continue
    clue_row = clue_key_to_row.get((cid, original_def), -1)
    if clue_row < 0:
        # Defensive: clue should always resolve. If this fires, something is wrong.
        continue
    clue_rows_out.append(clue_row)
    def_rows_out.append(wnex_val_word_to_row[def_wn])
    ans_rows_out.append(wnex_val_word_to_row[ans_wn])
    kept_eval_idx.append(i)

clue_rows_arr_wnex = np.array(clue_rows_out, dtype=np.int64)
def_rows_arr_wnex  = np.array(def_rows_out,  dtype=np.int64)
ans_rows_arr_wnex  = np.array(ans_rows_out,  dtype=np.int64)
kept_eval_idx_wnex = np.array(kept_eval_idx, dtype=np.int64)
n_kept_wnex = len(clue_rows_arr_wnex)
assemble_seconds = time.time() - t0

print(f"clues_val rows:                                  {n_eval_total:,}")
print(f"  rows where definition_wn is in wnex vocab:     {n_def_in_wnex:,} "
      f"({n_def_in_wnex / n_eval_total:.1%})")
print(f"  rows where answer_wn     is in wnex vocab:     {n_ans_in_wnex:,} "
      f"({n_ans_in_wnex / n_eval_total:.1%})")
print(f"  rows where BOTH are in wnex vocab (kept):      {n_kept_wnex:,} "
      f"({n_kept_wnex / n_eval_total:.1%})")
print(f"Assembly: {assemble_seconds:.1f}s")
print(f"\nNote: wnex_val covers only {len(wnex_val_word_to_row):,} of "
      f"{len(wndef_val_word_to_row):,} val vocabulary words "
      f"({len(wnex_val_word_to_row) / len(wndef_val_word_to_row):.1%}); "
      f"a small wnex evaluation set is expected.")

In [ ]:
# ============================================================
# §6b — T=0, T=1 distributions and ATE preview (wnex val)
# ============================================================
stock_T0_wnex, stock_T1_wnex = t0_t1(
    "g_stock", clue_rows_arr_wnex, def_rows_arr_wnex, ans_rows_arr_wnex,
    "f_common_wnex_val",
)
g1_T0_wnex, g1_T1_wnex = t0_t1(
    "g1", clue_rows_arr_wnex, def_rows_arr_wnex, ans_rows_arr_wnex,
    "f_common_wnex_val",
)

t0_t1_wnex_table = pd.DataFrame([
    dist_stats("g_stock T=0", stock_T0_wnex),
    dist_stats("g_stock T=1", stock_T1_wnex),
    dist_stats("g1 T=0",      g1_T0_wnex),
    dist_stats("g1 T=1",      g1_T1_wnex),
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(t0_t1_wnex_table.to_string(index=False))

ate_stock_wnex = float((stock_T1_wnex - stock_T0_wnex).mean())
ate_g1_wnex    = float((g1_T1_wnex    - g1_T0_wnex   ).mean())
print(f"\nATE preview (wnex, mean of T=1 - T=0 — formal test deferred to Stage 6):")
print(f"  g_stock ATE: {ate_stock_wnex:+.4f}")
print(f"  g1 ATE:      {ate_g1_wnex:+.4f}")
print(f"\nWndef ATE from §5b for cross-reference (different eval subset):")
print(f"  g_stock wndef ATE: {ate_stock_wndef:+.4f}  (n={n_kept_wndef:,} pairs)")
print(f"  g1 wndef ATE:      {ate_g1_wndef:+.4f}  (n={n_kept_wndef:,} pairs)")
print(f"  g_stock wnex  ATE: {ate_stock_wnex:+.4f}  (n={n_kept_wnex:,} pairs)")
print(f"  g1 wnex  ATE:      {ate_g1_wnex:+.4f}  (n={n_kept_wnex:,} pairs)")
print("Note: wndef and wnex ATEs above are computed on different (overlapping "
      "but not identical) subsets of clues_val. §6d makes the comparison on "
      "a matched subset of evaluation pairs.")

In [ ]:
# ============================================================
# §6c — T=0 / T=1 distribution figure (wnex)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

all_vals = np.concatenate([stock_T0_wnex, stock_T1_wnex,
                           g1_T0_wnex,    g1_T1_wnex])
bins = np.linspace(all_vals.min(), all_vals.max(), 70)

panels = [
    (axes[0], "g_stock", stock_T0_wnex, stock_T1_wnex),
    (axes[1], "g1",      g1_T0_wnex,    g1_T1_wnex),
]
for ax, model, T0, T1 in panels:
    ax.hist(T0, bins=bins, alpha=0.55, color="tab:green",
            label=f"T=0 (def vs ans), mean={T0.mean():.3f}")
    ax.hist(T1, bins=bins, alpha=0.55, color="tab:purple",
            label=f"T=1 (clue-def vs ans), mean={T1.mean():.3f}")
    ax.axvline(T0.mean(), color="tab:green",  linewidth=1.2, linestyle=":")
    ax.axvline(T1.mean(), color="tab:purple", linewidth=1.2, linestyle=":")
    ax.set_title(f"{model} — T=0 vs T=1 (wnex, {n_kept_wnex:,} eval pairs)")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("count")
    ax.legend()
fig.tight_layout()
fig_path = FIG_DIR / "05_t0_t1_wnex_distributions.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

### §6d — Matched ATE comparison (wndef vs wnex on identical pairs)

The §5b and §6b ATE numbers come from different clue subsets, so the
wndef-vs-wnex ATE difference could in principle be driven by which pairs
made it into each set rather than by the phrase format itself. Here we
restrict to the intersection — every pair that resolves under BOTH
wndef_val and wnex_val — and compute both ATEs on that single set. The
binding constraint is wnex_val (since wndef_val covers all val vocabulary
words), so the matched set is exactly the §6a kept set. The wndef ATE on
this subset is the cleanest comparison.

In [ ]:
# ============================================================
# §6d — Matched-pair ATE: same clues, two phrase formats
# ============================================================
# For each kept (i.e., wnex-eligible) row of clues_val, we already have its
# wnex row indices. We additionally need its wndef_val row indices for the
# matched comparison. Look those up by re-mapping definition_wn / answer_wn
# through the wndef_val dict (every val word is in vocab_wndef_val by
# construction, so these always resolve).
matched_def_rows_wndef = np.array([
    wndef_val_word_to_row[clues_val["definition_wn"].iloc[i]]
    for i in kept_eval_idx_wnex
], dtype=np.int64)
matched_ans_rows_wndef = np.array([
    wndef_val_word_to_row[clues_val["answer_wn"].iloc[i]]
    for i in kept_eval_idx_wnex
], dtype=np.int64)

# Recompute T=0/T=1 under wndef on the wnex-matched subset of clues.
stock_T0_wndef_m, stock_T1_wndef_m = t0_t1(
    "g_stock", clue_rows_arr_wnex,
    matched_def_rows_wndef, matched_ans_rows_wndef,
    "f_common_wndef_val",
)
g1_T0_wndef_m, g1_T1_wndef_m = t0_t1(
    "g1", clue_rows_arr_wnex,
    matched_def_rows_wndef, matched_ans_rows_wndef,
    "f_common_wndef_val",
)

ate_stock_wndef_m = float((stock_T1_wndef_m - stock_T0_wndef_m).mean())
ate_g1_wndef_m    = float((g1_T1_wndef_m    - g1_T0_wndef_m   ).mean())
# The wnex ATEs on the matched set are the same numbers as §6b (since the
# matched set IS the wnex eval set), but we reproduce them here for clarity.
ate_stock_wnex_m = ate_stock_wnex
ate_g1_wnex_m    = ate_g1_wnex

matched_ate_table = pd.DataFrame([
    {"Phrase format": "wndef",
     "g_stock ATE":   ate_stock_wndef_m,
     "g1 ATE":        ate_g1_wndef_m,
     "N pairs":       len(kept_eval_idx_wnex)},
    {"Phrase format": "wnex",
     "g_stock ATE":   ate_stock_wnex_m,
     "g1 ATE":        ate_g1_wnex_m,
     "N pairs":       len(kept_eval_idx_wnex)},
])

with pd.option_context("display.float_format", "{:+.4f}".format,
                       "display.width", 140):
    print(matched_ate_table.to_string(index=False))

---

## §7 — Representational Similarity Analysis (val-only)

Collapse detection in §4 looks at marginal properties of the embedding
space (mean similarity, variance spread). RSA asks a relational question:
did the *shape* of the similarity structure survive fine-tuning? If pairs
of words that were similar under `g_stock` are still the most similar
pairs under `g1` — even if all similarities have shifted up or down — the
Spearman correlation of the two pairwise-similarity vectors will be near
1. If fine-tuning fundamentally reorganized which words are similar to
which, the correlation will be low.

Combining §4 and §7 gives a two-axis reading:

- **Low RSA ρ + collapse signals** = the space compressed indiscriminately
- **Low RSA ρ + healthy dimensionality** = the space reorganized structurally
- **High RSA ρ** = fine-tuning was a global nudge, not a restructuring

We sample 1,000 words per phrase type (random_state=42) to keep the
1000×1000 cosine matrix small and the Spearman correlation fast. For the
wnex_val vocabulary (3,008 words) 1,000 is about a third of the words; for
wndef_val (26,152) it is less than 4%.

Val-only embeddings are used for the same model-selection discipline as
§4 (Decision 9).

In [ ]:
# ============================================================
# §7 — Representational Similarity Analysis (val-only)
# ============================================================
N_RSA = 1_000

def upper_triangle(mat):
    """Flatten the strict upper triangle of a square matrix."""
    n = mat.shape[0]
    iu = np.triu_indices(n, k=1)
    return mat[iu]

def pairwise_cosine_matrix(emb):
    """Return full NxN cosine similarity matrix for the given rows."""
    # Normalize once, then matmul — cheaper than rowwise-cosine over N^2 pairs.
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10
    normed = emb / norms
    return normed @ normed.T

rsa_rows = []
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    n_rows_phrase = embeddings[(MODEL_NAMES[0], phrase)].shape[0]
    # Sample the same row indices for both models (fixed seed per phrase).
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(n_rows_phrase, size=N_RSA, replace=False)

    # Build pairwise similarity matrix under each model, extract upper
    # triangle, then Spearman-correlate the two vectors.
    stock_mat = pairwise_cosine_matrix(embeddings[("g_stock", phrase)][sample_idx])
    g1_mat    = pairwise_cosine_matrix(embeddings[("g1",      phrase)][sample_idx])

    stock_ut = upper_triangle(stock_mat)
    g1_ut    = upper_triangle(g1_mat)
    rho, pval = spearmanr(stock_ut, g1_ut)

    rsa_rows.append({
        "Phrase type":     phrase,
        "N words sampled": N_RSA,
        "N pair values":   len(stock_ut),
        "Spearman rho":    float(rho),
        "p-value":         float(pval),
    })

rsa_df = pd.DataFrame(rsa_rows)
with pd.option_context("display.float_format", "{:.6f}".format,
                       "display.width", 140):
    print(rsa_df.to_string(index=False))

---

## §8 — Write results file

All key numbers are serialized to `outputs/05_model_evaluation-results.md`
so the Architect can review Stage 5 without re-running the notebook. The
file is strictly a summary — no interpretation, no PASS/FAIL verdicts.
Interpretation happens during the Stage 5 review and in the Stage 6
hypothesis-testing notebook that follows.

In [ ]:
# ============================================================
# Build outputs/05_model_evaluation-results.md
# ============================================================
def _fmt_cell(v, float_fmt="{:.4f}"):
    if isinstance(v, (float, np.floating)):
        return float_fmt.format(v)
    return str(v)

def df_to_md(df, float_fmt="{:.4f}"):
    """Render a DataFrame as a GitHub-flavored markdown table."""
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep    = "|" + "|".join(["---"] * len(cols)) + "|"
    rows = [
        "| " + " | ".join(_fmt_cell(v, float_fmt) for v in row) + " |"
        for row in df.itertuples(index=False, name=None)
    ]
    return chr(10).join([header, sep, *rows])

lines = []
lines.append("# Results: 05 — Model Evaluation (Pre-Hypothesis Testing)\n")
lines.append(f"**Date:** {date.today().isoformat()}  ")
lines.append(f"**Environment:** {env_label}\n")

lines.append("## Versions\n")
lines.append(f"- pandas:     {pd.__version__}")
lines.append(f"- numpy:      {np.__version__}")
lines.append(f"- scipy:      {scipy.__version__}")
lines.append(f"- matplotlib: {matplotlib.__version__}")
lines.append(f"- seaborn:    {seaborn.__version__}\n")

lines.append("## Scope\n")
lines.append("Canonical mean-pooling models only: `g_stock` and `g1`. "
             "The `_tokenspan` variants are out of scope per Decision 20.\n")
lines.append("Triplet accuracy (§2, §3) uses full-vocabulary wndef and wnex "
             "embeddings. Collapse (§4), T=0/T=1 (§5, §6), and RSA (§7) use "
             "validation-only embeddings (Decision 9 model-selection "
             "discipline).\n")

# §2
lines.append("## §2 — Validation triplet accuracy (wndef, full vocabulary)\n")
lines.append(f"Validation triplet file: `data/triplets/g1_val.csv` "
             f"({n_total:,} rows). Resolved against `vocabulary_wndef.csv` "
             f"({len(vocab_wndef):,} words).")
lines.append(f"- anchors resolved:   {n_total - n_miss_anchor:,} / {n_total:,}")
lines.append(f"- positives resolved: {n_total - n_miss_positive:,} / {n_total:,}")
lines.append(f"- negatives resolved: {n_total - n_miss_negative:,} / {n_total:,}")
lines.append(f"- all three resolved: {n_resolved_wndef:,} / {n_total:,} "
             f"({n_resolved_wndef / n_total:.1%}) — used for the accuracy "
             f"table below\n")
lines.append(df_to_md(triplet_table))
lines.append("")
lines.append(f"Figure: `outputs/figures/05_val_triplet_accuracy.png`\n")

# §3
lines.append("## §3 — Cross-f triplet accuracy (matched comparison)\n")
lines.append("Matched comparison: same triplet subset evaluated under wndef "
             "and wnex embeddings. A triplet enters the matched set if both "
             "its `answer_wn` and `distractor_wn` are present in "
             "`vocabulary_wnex.csv`.\n")
lines.append(f"Wnex resolution breakdown across {n_total:,} val triplets:")
lines.append(f"- both answer and distractor in wnex: {n_both_in:,} "
             f"({n_both_in / n_total:.1%}) — the matched subset")
lines.append(f"- answer in wnex, distractor not:     {n_pos_only:,} "
             f"({n_pos_only / n_total:.1%})")
lines.append(f"- distractor in wnex, answer not:     {n_neg_only:,} "
             f"({n_neg_only / n_total:.1%})")
lines.append(f"- neither in wnex:                    {n_neither:,} "
             f"({n_neither / n_total:.1%})\n")
lines.append(df_to_md(crossf_table))
lines.append("")
lines.append(f"Figure: `outputs/figures/05_crossf_triplet_accuracy.png`\n")

# §4a
lines.append("## §4a — Mean pairwise cosine among random word pairs (val-only)\n")
lines.append(f"Sampled {N_PAIRS:,} random distinct-row pairs per "
             f"(model, phrase) with random_state={RANDOM_STATE}. Same pairs "
             f"used for both models within a phrase type.\n")
lines.append(df_to_md(pairwise_df))
lines.append("")

# §4b
lines.append("## §4b — Embedding variance and effective dimensionality (val-only)\n")
lines.append(df_to_md(variance_df, float_fmt="{:.2f}"))
lines.append("")
lines.append("Figures: `outputs/figures/05_collapse_pairwise_cosine.png`, "
             "`outputs/figures/05_collapse_singular_values.png`\n")

# §5
lines.append("## §5 — T=0 and T=1 similarity distributions (wndef, val-only)\n")
lines.append(f"Evaluation pairs assembled from clues_val.csv ({n_eval_total:,} rows):")
lines.append(f"- dropped: (clue_id, definition) not in f_clue index: {n_missing_clue:,}")
lines.append(f"- dropped: definition_wn not in vocabulary_wndef_val:  {n_missing_def_wndef:,}")
lines.append(f"- dropped: answer_wn     not in vocabulary_wndef_val:  {n_missing_ans_wndef:,}")
lines.append(f"- kept:   {n_kept_wndef:,} ({n_kept_wndef / n_eval_total:.1%})\n")
lines.append(df_to_md(t0_t1_wndef_table))
lines.append("")
lines.append(f"ATE preview (deferred to Stage 6):")
lines.append(f"- g_stock ATE (mean of T=1 - T=0): {ate_stock_wndef:+.4f}")
lines.append(f"- g1 ATE      (mean of T=1 - T=0): {ate_g1_wndef:+.4f}\n")
lines.append(f"Figure: `outputs/figures/05_t0_t1_wndef_distributions.png`\n")

# §6
lines.append("## §6 — T=0 and T=1 similarity distributions (wnex, val-only)\n")
lines.append(f"Evaluation pairs assembled from clues_val.csv ({n_eval_total:,} rows):")
lines.append(f"- rows where definition_wn is in wnex vocab: {n_def_in_wnex:,} "
             f"({n_def_in_wnex / n_eval_total:.1%})")
lines.append(f"- rows where answer_wn     is in wnex vocab: {n_ans_in_wnex:,} "
             f"({n_ans_in_wnex / n_eval_total:.1%})")
lines.append(f"- rows where BOTH are in wnex vocab (kept):  {n_kept_wnex:,} "
             f"({n_kept_wnex / n_eval_total:.1%})\n")
lines.append(df_to_md(t0_t1_wnex_table))
lines.append("")
lines.append(f"ATE preview (deferred to Stage 6):")
lines.append(f"- g_stock ATE (mean of T=1 - T=0): {ate_stock_wnex:+.4f}")
lines.append(f"- g1 ATE      (mean of T=1 - T=0): {ate_g1_wnex:+.4f}\n")
lines.append("Note: the wndef ATE in §5 and the wnex ATE here are computed "
             "on different (overlapping but not identical) subsets of "
             "clues_val. §6d below makes the comparison on a matched subset.\n")
lines.append(f"Figure: `outputs/figures/05_t0_t1_wnex_distributions.png`\n")

# §6d
lines.append("### §6d — Matched ATE comparison (wndef vs wnex on identical pairs)\n")
lines.append(f"Restricted to the {len(kept_eval_idx_wnex):,} clues_val pairs "
             f"that resolve under both wndef_val and wnex_val. The wndef ATE "
             f"on this matched subset is the cleanest wndef-vs-wnex comparison.\n")
lines.append(df_to_md(matched_ate_table, float_fmt="{:+.4f}"))
lines.append("")

# §7
lines.append("## §7 — RSA (Spearman correlation of pairwise cosines, val-only)\n")
lines.append(df_to_md(rsa_df, float_fmt="{:.6f}"))
lines.append("")

results_path = OUTPUT_DIR / "05_model_evaluation-results.md"
results_path.write_text(chr(10).join(lines))
print(f"Wrote {results_path}")

---

## Summary

This notebook produced five diagnostic readings of `g1` before Stage 6
hypothesis testing:

1. **Validation triplet accuracy under wndef (§2, *updated*)** —
   Now uses the full-vocabulary `f_common_wndef` embeddings (53,930 words),
   resolving nearly all distractors that the prior val-only version
   (Decision 21) had to drop. The number of evaluated triplets jumped from
   ~27K to ~46K.
2. **Cross-f triplet accuracy (§3, *new*)** — The matched-comparison test
   from the design document Step B. Same triplet subset evaluated under
   both wndef and wnex embeddings; the binding constraint is the
   8,360-word wnex vocabulary. Two key comparisons fall out: (a) does g1
   wndef accuracy transfer to g1 wnex on the same triples? (b) does g1
   improve over g_stock on wnex despite never being trained on wnex
   phrases?
3. **Collapse detection (§4, *unchanged*)** — Same val-only data and
   methodology as the prior run; numbers serve as a consistency check on
   the refactor.
4. **T=0 / T=1 distributions under wndef (§5, *renamed*)** — Same val-only
   analysis as before, with the figure renamed to
   `05_t0_t1_wndef_distributions.png`.
5. **T=0 / T=1 distributions under wnex (§6, *new*)** — Parallel analysis
   on the wnex-eligible subset of clues_val. §6d does the matched ATE
   comparison on the intersection of pairs that resolve under both formats.
6. **RSA (§7, *unchanged*)** — Same val-only Spearman analysis as before.

**Outputs:**
- `outputs/05_model_evaluation-results.md` — all numerical results
- `outputs/figures/05_val_triplet_accuracy.png` (updated, ~46K triplets)
- `outputs/figures/05_crossf_triplet_accuracy.png` (new)
- `outputs/figures/05_collapse_pairwise_cosine.png` (unchanged content)
- `outputs/figures/05_collapse_singular_values.png` (unchanged content)
- `outputs/figures/05_t0_t1_wndef_distributions.png` (renamed from `05_t0_t1_distributions.png`)
- `outputs/figures/05_t0_t1_wnex_distributions.png` (new)

No new data artifacts are written — the notebook is read-only with respect
to the data directory.

**Runtime:** A few seconds on CPU, dominated by the two SVDs over the
26,152-row `f_common_wndef_val` matrices in §4b.